# Speed comparison

How fast fse trains depends mostly on three choices: the input class, the number of workers and the storage format of the vectors. This notebook times `train` end to end for each of them on one machine. The numbers vary with the machine and its load; for careful measurements in fresh processes, with the load average recorded, use `benchmarks/bench_train.py` in the repository.

In [ ]:
import csv
import gzip
import logging
import os
import platform
import sys
import tempfile
import time

import gensim.downloader as api
import pandas as pd

from fse import SIF, Average, CorpusFile, IndexedList, TextList, Vectors
from fse.models.average import FAST_VERSION

logging.basicConfig(level=logging.WARNING)
print(platform.platform(), platform.processor(), f"{os.cpu_count()} logical CPUs")
print("compiled routines:", FAST_VERSION == 1)
print("load average:", os.getloadavg())

The data are the Quora duplicate questions from gensim-data, repeated four times, and the vectors are `glove-wiki-gigaword-100`.

In [ ]:
path = api.load("quora-duplicate-questions", return_path=True)  # its own loader needs an old smart_open
csv.field_size_limit(sys.maxsize)
with gzip.open(path, "rt", encoding="utf-8") as f:
    data = list(csv.DictReader(f, delimiter="\t"))
questions = [q for d in data for q in (d["question1"], d["question2"])] * 4
glove = Vectors.from_pretrained("glove-wiki-gigaword-100")
print(f"{len(questions):,} sentences")

In [ ]:
def sentences_per_second(model, sentences, repeat=3):
    """Best of `repeat` trainings, in sentences per second."""
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        model.train(sentences)
        best = min(best, time.perf_counter() - start)
    return len(questions) / best

## Input classes

`IndexedList` needs the sentences split into Python lists first, `TextList` splits the strings in compiled code, and `CorpusFile` reads them from a file. With several workers, all three are read by every worker for its own part of the input.

In [ ]:
split = IndexedList([q.split() for q in questions])
text = TextList(questions)
tmp = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8")
tmp.write("".join(q.replace("\n", " ") + "\n" for q in questions))
tmp.close()
corpus = CorpusFile(tmp.name)

inputs = {"IndexedList": split, "TextList": text, "CorpusFile": corpus}
workers = sorted({1, 2, 4, os.cpu_count()})
rows = []
for name, sentences in inputs.items():
    for w in workers:
        rows.append({"input": name, "workers": w, "sentences/s": sentences_per_second(Average(glove, workers=w), sentences)})
pd.DataFrame(rows).pivot(index="input", columns="workers", values="sentences/s").round(-3)

## Models

SIF also learns and removes principal components, which costs a second pass over the sentence vectors.

In [ ]:
rows = []
for name, cls, kwargs in [("Average", Average, {}), ("SIF", SIF, {"lang_freq": "en"})]:
    for w in workers:
        rows.append({"model": name, "workers": w, "sentences/s": sentences_per_second(cls(glove, workers=w, **kwargs), text)})
pd.DataFrame(rows).pivot(index="model", columns="workers", values="sentences/s").round(-3)

## Storage formats

float16 and int8 vectors take half or a quarter of the memory. Where fse reads the codes with vector instructions (AVX2 on x86-64, NEON on arm64), averaging int8 word vectors also gets somewhat faster.

In [ ]:
rows = []
w = os.cpu_count()
for wv_dtype in ["float32", "float16", "int8"]:
    for sv_dtype in ["float32", "int8"]:
        model = Average(glove, workers=w, wv_dtype=wv_dtype, sv_dtype=sv_dtype)
        rows.append({"wv_dtype": wv_dtype, "sv_dtype": sv_dtype, "sentences/s": sentences_per_second(model, text)})
pd.DataFrame(rows).pivot(index="wv_dtype", columns="sv_dtype", values="sentences/s").round(-3)

In [ ]:
os.unlink(tmp.name)